# a) Representation and Fitness

In [78]:
import random

COURSES = 6
ROOMS = 3
SLOTS = 4

def random_chromosome():
    return [(random.randint(0, ROOMS-1), random.randint(0, SLOTS-1))
            for _ in range(COURSES)]

def count_conflicts(chromosome):
    conflicts = 0

    for i in range(len(chromosome)):
        for j in range(i+1, len(chromosome)):
            if chromosome[i] == chromosome[j]:
                conflicts += 1

    return conflicts

def fitness(chromosome):
    return 100 - (10 * count_conflicts(chromosome))

In [79]:
def print_initial_samples():
    print("Random Chromosomes:\n")
    print(f"{'Chromosome':<50}{'Conflicts':<10}{'Fitness'}")

    for _ in range(5):
        c = random_chromosome()
        conf = count_conflicts(c)
        fit = fitness(c)

        print(f"{str(c):<50}{conf:<10}{fit}")

print_initial_samples()

Random Chromosomes:

Chromosome                                        Conflicts Fitness
[(2, 3), (0, 1), (1, 0), (2, 1), (0, 2), (0, 2)]  1         90
[(2, 1), (2, 1), (2, 0), (1, 0), (2, 2), (0, 2)]  1         90
[(0, 3), (1, 2), (0, 3), (0, 2), (0, 1), (2, 1)]  1         90
[(1, 2), (0, 1), (1, 3), (1, 1), (2, 3), (2, 2)]  0         100
[(0, 1), (1, 1), (2, 2), (1, 2), (0, 3), (2, 1)]  0         100


# b) Crossover and Repair and Mutation

In [80]:
def crossover(p1, p2, point):
    return (
        p1[:point] + p2[point:],
        p2[:point] + p1[point:]
    )

def repair(chromosome):
    new_chrom = chromosome[:]

    for i in range(len(new_chrom)):
        while True:
            conflict = False

            for j in range(len(new_chrom)):
                if i != j and new_chrom[i] == new_chrom[j]:
                    conflict = True
                    break

            if not conflict:
                break

            new_chrom[i] = (random.randint(0, ROOMS-1),
                           random.randint(0, SLOTS-1))

    return new_chrom

def mutate(chromosome, pm):
    new = []
    for gene in chromosome:
        if random.random() < pm:
            new.append((random.randint(0, ROOMS-1),
                        random.randint(0, SLOTS-1)))
        else:
            new.append(gene)
    return new

In [81]:
def demo_crossover_conflict():
    p1 = [(0,0),(1,1),(2,2),(0,1),(1,2),(2,3)]
    p2 = [(0,0),(1,1),(2,2),(0,1),(1,2),(2,3)]

    c1, c2 = crossover(p1, p2, 3)

    print("\nCrossover Demo:")
    print("Parent 1:", p1)
    print("Parent 2:", p2)
    print("Offspring 1:", c1, "Conflicts:", count_conflicts(c1))
    print("Offspring 2:", c2, "Conflicts:", count_conflicts(c2))

demo_crossover_conflict()


Crossover Demo:
Parent 1: [(0, 0), (1, 1), (2, 2), (0, 1), (1, 2), (2, 3)]
Parent 2: [(0, 0), (1, 1), (2, 2), (0, 1), (1, 2), (2, 3)]
Offspring 1: [(0, 0), (1, 1), (2, 2), (0, 1), (1, 2), (2, 3)] Conflicts: 0
Offspring 2: [(0, 0), (1, 1), (2, 2), (0, 1), (1, 2), (2, 3)] Conflicts: 0


In [82]:
def demo_repair():
    bad = [(0,0),(0,0),(1,1),(1,1),(2,2),(2,2)]

    print("\nRepair Demo:")
    print("Before:", bad, "Conflicts:", count_conflicts(bad))

    fixed = repair(bad)

    print("After :", fixed, "Conflicts:", count_conflicts(fixed))

demo_repair()


Repair Demo:
Before: [(0, 0), (0, 0), (1, 1), (1, 1), (2, 2), (2, 2)] Conflicts: 3
After : [(2, 0), (0, 0), (0, 1), (1, 1), (0, 2), (2, 2)] Conflicts: 0


# c) Full FA (Tournament Selection)

In [83]:
def tournament_select(pop):
    a, b = random.sample(pop, 2)
    return a if fitness(a) > fitness(b) else b

def run_scheduling_ga(pop_size, generations, pm):
    population = [random_chromosome() for _ in range(pop_size)]

    best_per_gen = []

    for gen in range(generations):
        population.sort(key=fitness, reverse=True)
        best = population[0]
        best_fit = fitness(best)

        best_per_gen.append(best_fit)

        if count_conflicts(best) == 0:
            print(f"\nSolution found at generation {gen}: {best}")
            return best, best_fit, best_per_gen

        new_pop = []

        while len(new_pop) < pop_size:
            p1 = tournament_select(population)
            p2 = tournament_select(population)

            point = random.randint(1, COURSES-1)
            c1, c2 = crossover(p1, p2, point)

            c1 = repair(c1)
            c2 = repair(c2)

            c1 = mutate(c1, pm)
            c2 = mutate(c2, pm)

            new_pop.extend([c1, c2])

        population = new_pop[:pop_size]

    population.sort(key=fitness, reverse=True)
    best = population[0]

    return best, fitness(best), best_per_gen

In [84]:
print("\n=== Scheduling GA Run ===")

best, best_fit, history = run_scheduling_ga(
    pop_size=20,
    generations=50,
    pm=0.1
)

print("\nBest Schedule:", best)
print("Conflicts:", count_conflicts(best))
print("Fitness:", best_fit)


=== Scheduling GA Run ===

Solution found at generation 0: [(1, 0), (0, 0), (2, 3), (1, 1), (0, 2), (0, 3)]

Best Schedule: [(1, 0), (0, 0), (2, 3), (1, 1), (0, 2), (0, 3)]
Conflicts: 0
Fitness: 100
